# IOP Simulation Study (Python + NumPy + Numba)
Optimized notebook scaffold converted from the uploaded R script.

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from scipy.stats import norm, shapiro
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

RNG_SEED = 2025
np.random.seed(RNG_SEED)


In [ ]:
@njit(cache=True)
def weighted_gini(y, w):
    idx = np.argsort(y)
    ys = y[idx]
    ws = w[idx]
    ws = ws / ws.sum()
    F = np.cumsum(ws) - ws / 2.0
    mu = np.sum(ws * ys)
    if mu < 1e-12:
        return 0.0
    g = 1.0 - 2.0 * np.sum(ws * ys * (1.0 - F)) / mu
    return max(g, 0.0)


In [ ]:
def compute_iop(y, X, w):
    logy = np.log(y)
    XtW = X.T * w
    beta = np.linalg.solve(XtW @ X, XtW @ logy)
    xhat = np.exp(X @ beta)

    omega = weighted_gini(y, w)
    omega_a = weighted_gini(xhat, w)
    omega_r = np.nan if omega < 1e-12 else omega_a / omega

    return {
        "omega": omega,
        "omega_a": omega_a,
        "omega_r": omega_r,
        "beta": beta,
    }


## NOTE

This notebook is a starter framework. The remaining sections to implement are:
1. Two-stage stratified sampling
2. Jackknife variance estimator
3. Main Monte Carlo simulation
4. Power study
5. Tables and figures